# PCB remaining useful life evaluation

Select a GPU under Runtime → Change runtime type. Run the cell and select `pcb-rul-kaggle-v2-checkpoint.zip` when prompted. The cell loads the frozen models, runs the validation checks and downloads the final scores and plots.

In [ ]:
from hashlib import file_digest, sha256
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZipFile

from google.colab import files

work = Path("/content/pcb-rul-evaluation")
outputs = work / "pcb_rul/outputs"
checkpoint_name = "pcb-rul-kaggle-v2-checkpoint.zip"
report = outputs / "test_report.json"

if not report.is_file():
    subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], check=True)
    if (outputs / "test_access.json").exists():
        raise RuntimeError("An evaluation already opened the test split in this runtime. Keep the files and share the error.")

    checkpoint = Path("/content") / checkpoint_name
    if not checkpoint.is_file():
        print("Select the checkpoint ZIP when Colab asks.")
        uploaded = files.upload()
        if checkpoint_name not in uploaded:
            raise ValueError(f"Select {checkpoint_name}")
        checkpoint.write_bytes(uploaded[checkpoint_name])
        del uploaded

    work.mkdir(exist_ok=True)
    with ZipFile(checkpoint) as archive:
        manifest = archive.read("sha256.json")
        if sha256(manifest).hexdigest() != 'e1ed2c16a8873cdd005e7208100d4a3d0110e916c5b1007236f031868c5944ba':
            raise ValueError("This is not the verified Kaggle version 2 checkpoint.")
        for name, digest in json.loads(manifest).items():
            with archive.open(name) as stream:
                if file_digest(stream, "sha256").hexdigest() != digest:
                    raise ValueError(f"Checkpoint checksum mismatch for {name}")
            archive.extract(name, work)
    print("Checkpoint verified. Setting up Python 3.11.")

    env = os.environ.copy()
    env.update(MPLBACKEND="Agg", MPLCONFIGDIR="/tmp/pcb-rul-matplotlib",
               PCB_RUL_LIGHTGBM_DEVICE="cpu", CUBLAS_WORKSPACE_CONFIG=":4096:8",
               OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1")
    uv = shutil.which("uv")
    if uv is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
        uv = shutil.which("uv")
    venv = work / ".venv"
    subprocess.run([uv, "venv", "--python", "3.11", str(venv)], check=True, env=env)
    python = str(venv / "bin/python")
    packages = [
        "numpy==2.4.6", "pandas==3.0.5", "scikit-learn==1.9.0", "lightgbm==4.7.0",
        "torch==2.7.1", "optuna==5.0.0", "matplotlib==3.11.1", "scipy==1.17.1",
    ]
    subprocess.run([uv, "pip", "install", "--python", python, *packages], check=True, env=env)
    subprocess.run([python, "-c",
                    "import sys, torch; assert sys.version_info[:2] == (3, 11); "
                    "assert torch.cuda.is_available(), 'Select a GPU runtime'; "
                    "print('PyTorch GPU:', torch.cuda.get_device_name(0))"],
                   check=True, env=env)
    print("Running validation checks and scoring the saved models.", flush=True)
    subprocess.run([python, "-u", str(work / "pcb_rul/evaluate.py")], cwd=work, env=env, check=True)

result = json.loads(report.read_text())
assert result["synthetic"] and result["test_units"] == 150
result_zip = Path("/content/pcb-rul-final-results.zip")
with ZipFile(result_zip, "w") as archive:
    for name in ("test_report.json", "test_comparison.csv", "test_by_seed.csv",
                 "test_access.json", "ensemble.json", "five_test_units.png",
                 "error_by_life_stage.png", "predicted_vs_true.png"):
        archive.write(outputs / name, name)
print(f"Saved {result_zip}")
files.download(str(result_zip))
